<a href="https://colab.research.google.com/github/kiran58-max/toxicity-moe-ablation/blob/main/DistilBERT_MoE_Ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import DistilBertModel, DistilBertTokenizer
from datasets import load_dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np
import random

SEED = 42

def set_seed():
    """Call this immediately before EVERY model instantiation."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_datasets():
    datasets_dict = {}
    print("Loading Davidson...")
    url = "https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv"
    df_dav = pd.read_csv(url)
    df_dav['binary_label'] = df_dav['class'].apply(lambda x: 0 if x == 2 else 1)
    df_dav_sampled = df_dav.sample(n=5000, random_state=SEED).reset_index(drop=True)
    datasets_dict['Davidson'] = {'texts': df_dav_sampled['tweet'].values, 'labels': df_dav_sampled['binary_label'].values}

    print("Loading HatEval...")
    hateval = load_dataset("cardiffnlp/tweet_eval", "hate")
    df_hat = pd.concat([pd.DataFrame(hateval['train']), pd.DataFrame(hateval['test'])])
    df_hat_sampled = df_hat.sample(n=5000, random_state=SEED).reset_index(drop=True)
    datasets_dict['HatEval'] = {'texts': df_hat_sampled['text'].values, 'labels': df_hat_sampled['label'].values}
    return datasets_dict

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
BATCH_SIZE = 16

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    _, _, f1_mac, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return acc, f1_mac

class BertMoEConfigurable(nn.Module):
    def __init__(self, num_experts=4, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        for name, param in self.bert.named_parameters():
            param.requires_grad = True if ("layer.5" in name or "layer.4" in name) else False
        self.experts = nn.ModuleList([nn.Sequential(
            nn.Linear(768, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 768)
        ) for _ in range(num_experts)])
        self.router = nn.Sequential(nn.Linear(768, num_experts), nn.Softmax(dim=-1))
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        routing_weights = self.router(out)
        expert_outputs = torch.stack([expert(out) for expert in self.experts], dim=1)
        combined = torch.sum(routing_weights.unsqueeze(-1) * expert_outputs, dim=1)
        final_rep = (out + combined) if self.use_residual else combined
        return self.classifier(final_rep), routing_weights

class BertMoEScaled(nn.Module):
    def __init__(self, num_experts=16, hidden_dim=256):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        for name, param in self.bert.named_parameters():
            param.requires_grad = True if ("layer.5" in name or "layer.4" in name) else False
        self.experts = nn.ModuleList([nn.Sequential(
            nn.Linear(768, hidden_dim), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden_dim, 768)
        ) for _ in range(num_experts)])
        self.router = nn.Sequential(nn.Linear(768, num_experts), nn.Softmax(dim=-1))
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        routing_weights = self.router(out)
        expert_outputs = torch.stack([expert(out) for expert in self.experts], dim=1)
        combined = torch.sum(routing_weights.unsqueeze(-1) * expert_outputs, dim=1)
        final_rep = out + combined
        return self.classifier(final_rep), routing_weights

def train_moe_variant(model, train_loader, val_loader, class_weights, differential_lr, epochs):
    model = model.to(device)
    if differential_lr:
        optimizer = AdamW([
            {'params': model.bert.parameters(), 'lr': 2e-5},
            {'params': model.experts.parameters(), 'lr': 1e-4},
            {'params': model.router.parameters(), 'lr': 1e-4},
            {'params': model.classifier.parameters(), 'lr': 1e-4}
        ])
    else:
        optimizer = AdamW(model.parameters(), lr=2e-5)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            b_input_ids, b_mask, b_labels = [t.to(device) for t in batch]
            optimizer.zero_grad()
            outputs, _ = model(b_input_ids, b_mask)
            loss = criterion(outputs, b_labels)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_labels, all_routing = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            b_input_ids, b_mask, b_labels = [t.to(device) for t in batch]
            outputs, routing_w = model(b_input_ids, b_mask)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(b_labels.cpu().numpy())
            all_routing.append(routing_w.cpu().numpy())

    metrics = compute_metrics(all_labels, all_preds)
    routing_mat = np.concatenate(all_routing, axis=0)
    mean_util = np.mean(routing_mat, axis=0)
    agg_entropy = -np.sum(mean_util * np.log(mean_util + 1e-9))
    row_entropies = -np.sum(routing_mat * np.log(routing_mat + 1e-9), axis=1)
    per_ex_entropy = np.mean(row_entropies)
    return metrics, mean_util, agg_entropy, per_ex_entropy

datasets = load_datasets()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

ablation_configs = {
    'A_broken_baseline':  dict(use_residual=False, differential_lr=False, epochs=3),
    'B_residual_only':    dict(use_residual=True,  differential_lr=False, epochs=3),
    'C_residual_plus_lr': dict(use_residual=True,  differential_lr=True,  epochs=3),
    'D_final_config':     dict(use_residual=True,  differential_lr=True,  epochs=6),
}

for ds_name, data in datasets.items():
    print(f"\n{'='*50}\nDATASET: {ds_name}\n{'='*50}")
    texts, labels = data['texts'], data['labels']

    ablation_results = {name: {'acc': [], 'f1': []} for name in ablation_configs}
    scaled_results = {'acc': [], 'f1': [], 'agg_ent': [], 'per_ex_ent': []}

    for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels), 1):
        print(f"\n--- Fold {fold}/5 ---")
        train_texts, val_texts = texts[train_idx], texts[val_idx]
        train_labels, val_labels = labels[train_idx], labels[val_idx]

        train_encodings = tokenizer(train_texts.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
        val_encodings = tokenizer(val_texts.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
        train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], torch.tensor(train_labels))
        val_dataset = TensorDataset(val_encodings['input_ids'], val_encodings['attention_mask'], torch.tensor(val_labels))
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
        class_weights = torch.tensor([
            len(train_labels) / (2.0 * sum(train_labels == 0)),
            len(train_labels) / (2.0 * sum(train_labels == 1))
        ], dtype=torch.float)

        # EXPERIMENT 1: Ablation table — RESEEDED before each variant
        for name, cfg in ablation_configs.items():
            set_seed()  # <-- THE FIX: identical init/dropout/shuffle state for every variant
            model = BertMoEConfigurable(use_residual=cfg['use_residual'])
            metrics, _, _, _ = train_moe_variant(
                model, train_loader, val_loader, class_weights,
                differential_lr=cfg['differential_lr'], epochs=cfg['epochs']
            )
            ablation_results[name]['acc'].append(metrics[0])
            ablation_results[name]['f1'].append(metrics[1])
            print(f"  [Ablation:{name}] Acc={metrics[0]:.4f}, F1={metrics[1]:.4f}")

        # EXPERIMENT 2: Scaled MoE — also reseeded for consistency
        set_seed()
        scaled_model = BertMoEScaled(num_experts=16, hidden_dim=256)
        s_metrics, s_util, s_agg_ent, s_per_ex_ent = train_moe_variant(
            scaled_model, train_loader, val_loader, class_weights,
            differential_lr=True, epochs=6
        )
        scaled_results['acc'].append(s_metrics[0])
        scaled_results['f1'].append(s_metrics[1])
        scaled_results['agg_ent'].append(s_agg_ent)
        scaled_results['per_ex_ent'].append(s_per_ex_ent)
        print(f"  [Scaled K=16/256] Acc={s_metrics[0]:.4f}, F1={s_metrics[1]:.4f}, "
              f"AggEnt={s_agg_ent:.4f}, PerExEnt={s_per_ex_ent:.4f}")

        torch.cuda.empty_cache()

    print(f"\n*** FINAL RESULTS: {ds_name} ***")
    print("\n-- Experiment 1: Ablation Table --")
    for name, res in ablation_results.items():
        acc_arr, f1_arr = np.array(res['acc']), np.array(res['f1'])
        print(f"{name}: Acc={acc_arr.mean():.4f} (±{acc_arr.std(ddof=1):.4f}) | "
              f"F1={f1_arr.mean():.4f} (±{f1_arr.std(ddof=1):.4f})")

    print("\n-- Experiment 2: Scaled MoE (K=16, hidden=256) --")
    acc_arr, f1_arr = np.array(scaled_results['acc']), np.array(scaled_results['f1'])
    print(f"Acc={acc_arr.mean():.4f} (±{acc_arr.std(ddof=1):.4f}) | "
          f"F1={f1_arr.mean():.4f} (±{f1_arr.std(ddof=1):.4f}) | "
          f"AggEnt={np.mean(scaled_results['agg_ent']):.4f} | "
          f"PerExEnt={np.mean(scaled_results['per_ex_ent']):.4f}")

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading Davidson...
Loading HatEval...


...

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [Ablation:D_final_config] Acc=0.7630, F1=0.7619


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [Scaled K=16/256] Acc=0.7620, F1=0.7542, AggEnt=2.6915, PerExEnt=2.5530

*** FINAL RESULTS: HatEval ***

-- Experiment 1: Ablation Table --
A_broken_baseline: Acc=0.7454 (±0.0274) | F1=0.7426 (±0.0256)
B_residual_only: Acc=0.7534 (±0.0212) | F1=0.7511 (±0.0199)
C_residual_plus_lr: Acc=0.7526 (±0.0190) | F1=0.7504 (±0.0178)
D_final_config: Acc=0.7580 (±0.0138) | F1=0.7497 (±0.0146)

-- Experiment 2: Scaled MoE (K=16, hidden=256) --
Acc=0.7670 (±0.0147) | F1=0.7608 (±0.0144) | AggEnt=2.7249 | PerExEnt=2.6099
